# Bankruptcy Filing Risk Predictor

**Course:** BA870-AC820  
**Data:** SEC EDGAR 10-K / 10-Q filings + yfinance market data  
**Scope:** educational/prototype screening tool for estimated bankruptcy filing risk

---

## Project Overview

This notebook documents the final methodology behind the Streamlit dashboard. The project estimates **bankruptcy filing risk** by comparing current companies to historical bankruptcy-filing examples and public-company controls.

Important interpretation notes:

- The positive label is **bankruptcy filing risk**, not final liquidation or confirmed permanent failure.
- Model probabilities are **estimated filing-risk similarity scores**, not certainty.
- This is an educational/prototype screening tool and **not investment advice**.
- Reported validation metrics describe prototype screening performance on a curated dataset; they may not generalize to broader real-world samples.

## Setup — Imports & Configuration

In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)

DATA_DIR = Path('data/processed')

---

## Data Sources

The project uses public/free data sources:

1. **SEC EDGAR XBRL Company Facts API**  
   Used for 10-K / 10-Q financial statement fields such as assets, liabilities, revenue, net income, EBIT, retained earnings, current assets, and current liabilities.

2. **yfinance**  
   Used for market data where available. Historical delisted ticker coverage is limited, so the stock-price signal section uses only verified examples.

3. **S&P 500 constituent list**  
   Used for current public-company scoring and public controls.

Current committed training artifact:

- **178 companies**
- **44 bankruptcy-filing positives**
- **134 controls**

For filing-positive companies, the pipeline uses the **final pre-filing 10-K where available**, avoiding post-filing or successor filings.

In [ ]:
metadata = json.loads((DATA_DIR / 'training_metadata.json').read_text())
metadata

In [ ]:
features = pd.read_csv(DATA_DIR / 'features.csv')
predictions = pd.read_csv(DATA_DIR / 'predictions.csv')
comparison = pd.read_csv(DATA_DIR / 'model_comparison.csv')

label_col = 'bankruptcy_filing' if 'bankruptcy_filing' in features.columns else 'bankrupt'
print(f"Feature matrix: {len(features)} rows × {features.shape[1]} columns")
print(f"Filing positives: {int(features[label_col].sum())}")
print(f"Controls: {int((features[label_col] == 0).sum())}")
features[['ticker','name','sector',label_col,'source_filing_end','source_filing_filed']].head()

---

## Feature Engineering

Raw financial statement values are transformed into ratios so companies of different sizes can be compared.

Main feature groups:

- **Liquidity:** current ratio, working capital / assets
- **Leverage:** debt-to-equity, equity / liabilities
- **Profitability:** return on assets, net margin, EBIT / assets
- **Operating efficiency:** revenue / assets
- **Altman Z-score variables:** `z_x1` through `z_x5`
- **Fraud-risk indicators:** simplified rule-based red flags inspired by Beneish-style concepts

Missing values are handled inside sklearn Pipelines during model evaluation/training to reduce preprocessing leakage.

In [ ]:
MODEL_FEATURES = [
    'current_ratio', 'debt_to_equity', 'return_on_assets',
    'net_margin', 'interest_coverage',
    'z_x1', 'z_x2', 'z_x3', 'z_x4', 'z_x5',
]

features[MODEL_FEATURES + ['z_score','fraud_risk_score']].describe().T[['count','mean','std','min','max']]

In [ ]:
ratio_summary = features.groupby(label_col)[MODEL_FEATURES].median(numeric_only=True).T
ratio_summary.columns = ['Control median', 'Filing-positive median'] if list(ratio_summary.columns) == [0, 1] else ratio_summary.columns
ratio_summary

### Altman Z-Score and Fraud Risk Score

Altman Z-Score and Fraud Risk Score are included as **benchmark/rule-based indicators**. They are useful comparison points, but they are not supervised ML models in this project.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

features['z_score'].dropna().hist(ax=axes[0], bins=20, color='#4C78A8')
axes[0].axvline(1.81, color='red', linestyle='--', label='Distress threshold')
axes[0].set_title('Altman Z-Score Distribution')
axes[0].set_xlabel('Z-score')
axes[0].legend()

features['fraud_risk_score'].hist(ax=axes[1], bins=range(0, 7), color='#F58518', align='left')
axes[1].set_title('Fraud Risk Score Distribution')
axes[1].set_xlabel('Number of red flags')

plt.tight_layout()
plt.show()

---

## Modeling

The final app compares four approaches:

| Model / Indicator | Type | Role |
|---|---|---|
| Logistic Regression | Supervised model | Main S&P 500 scoring model |
| Random Forest | Supervised model | Comparison model; probabilities are calibrated |
| Altman Z-Score | Rule-based benchmark | Classic bankruptcy-risk indicator |
| Fraud Risk Score | Rule-based indicator | Heuristic red-flag count |

The deployed S&P 500 scoring model currently uses **Logistic Regression**.

---

## Validation Design

To make evaluation more conservative and presentation-defensible, the training script reports:

- **Grouped cross-validation** so the same ticker/company group cannot appear in both train and validation folds.
- Current artifact check: **one row per ticker**, 178 validation groups, no repeated ticker groups.
- A separate **45-row stratified holdout test set**.
- **Majority-class baseline accuracy**.
- **Brier score** for probability quality.
- Calibrated Random Forest probabilities using `CalibratedClassifierCV`.

These metrics should be interpreted as **prototype screening performance**, not real-world predictive accuracy.

In [ ]:
metrics = comparison.copy()
for col in ['Accuracy','Precision','Recall']:
    metrics[col] = metrics[col].map(lambda x: f'{x:.1%}')
for col in ['ROC-AUC','Brier']:
    metrics[col] = metrics[col].map(lambda x: f'{x:.3f}')
metrics

In [ ]:
plot_df = comparison[comparison['Evaluation'].isin(['Group CV','Holdout','Holdout baseline'])].copy()
fig, ax = plt.subplots(figsize=(10, 5))
labels = plot_df['Model'] + ' — ' + plot_df['Evaluation']
ax.barh(labels, plot_df['Accuracy'], color='#4C78A8')
ax.set_xlim(0, 1)
ax.set_xlabel('Accuracy')
ax.set_title('Prototype Screening Performance')
for i, v in enumerate(plot_df['Accuracy']):
    ax.text(v + 0.01, i, f'{v:.1%}', va='center')
plt.tight_layout()
plt.show()

### Current Key Metrics

- Logistic Regression Group CV: **89.9% accuracy**, Brier **0.094**
- Logistic Regression Holdout: **95.6% accuracy**, Brier **0.077**
- Random Forest Group CV: **92.7% accuracy**, Brier **0.053**
- Random Forest Holdout: **97.8% accuracy**, Brier **0.031**
- Majority-class holdout baseline: **75.6% accuracy**

The high supervised-model metrics likely reflect the curated educational dataset and should not be overclaimed.

In [ ]:
fi = pd.read_csv(DATA_DIR / 'feature_importance.csv').sort_values('importance')
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi['feature'], fi['importance'], color='#72B7B2')
ax.set_title('Logistic Regression Feature Importance')
ax.set_xlabel('Absolute coefficient')
plt.tight_layout()
plt.show()

---

## Dashboard Summary

The final Streamlit app includes:

1. **S&P 500 Leaderboard** — current filing-risk scores and period-over-period deltas.
2. **Watchlist** — top current S&P 500 companies ranked by estimated filing risk, with risk category and financial drivers.
3. **Company Lookup** — company-level filing metadata, risk score, and key ratios.
4. **Model Validation** — grouped CV, holdout metrics, baselines, benchmarks, Brier score, confusion matrices, and feature importance.
5. **Pre-filing stock signal demo** — verified yfinance examples for `PCG` and `CZR` only.

The app labels model outputs as estimated filing-risk similarity scores and includes not-investment-advice language.

In [ ]:
sp500_path = DATA_DIR / 'sp500_predictions.csv'
if sp500_path.exists():
    sp500 = pd.read_csv(sp500_path)
    scored = sp500[sp500['data_available'] == True].copy()
    print(f"S&P 500 rows with data: {len(scored)}")
    display_cols = ['ticker','name','sector','current_prob','risk_bucket','delta_prob','current_period_end']
    display(scored.sort_values('current_prob', ascending=False)[display_cols].head(10))
else:
    print('S&P 500 predictions file not found. Run refresh_sp500.py to generate it.')

---

## Optional Long-Running Rebuild Cells

The following commands rebuild data/model/dashboard artifacts. They call SEC EDGAR and yfinance and may take several minutes, so they are intentionally not run as part of the normal notebook flow.

In [ ]:
# Optional / long-running:
# !python collect_data.py --control-limit 120
# !python feature_engineering.py
# !python train_models.py
# !python refresh_sp500.py
# !streamlit run dashboard/app.py

---

## Limitations

- This is a **curated educational dataset**, not a production credit-risk panel.
- Metrics may not generalize to larger real-world samples.
- Model probabilities are similarity scores, not certainty.
- Positive labels indicate bankruptcy filing risk, not final liquidation.
- Broader historical market-signal analysis needs verified ticker mappings because many filing-company symbols are delisted or unavailable through yfinance.
- The app is a prototype screening tool and is not investment advice.

## Conclusion

The project demonstrates a public-data workflow for bankruptcy filing risk screening: collect SEC financial statement data, engineer interpretable financial ratios, compare supervised models against rule-based benchmarks, and present current S&P 500 screening results in a Streamlit dashboard. The final methodology is more conservative than a simple backtest because it includes grouped cross-validation, a holdout set, a majority-class baseline, benchmark indicators, and probability-quality reporting.